# Optimalisasi Model Prophet untuk Prediksi Penjualan Rokok Mingguan
## Studi Kasus: PT XYZ Provinsi Sulawesi Tenggara — Data 2022–2025

Notebook final revisi ini dirancang untuk:

1. membaca data mingguan lengkap tahun 2022–2025;
2. menggunakan UMP Provinsi Sulawesi Tenggara;
3. menggunakan indeks kumulatif kebijakan CHT nasional sebagai proxy, bukan tarif per batang atau HJE;
4. membandingkan Prophet tanpa dan dengan regressor eksternal;
5. menguji yearly seasonality karena tersedia beberapa siklus tahunan penuh;
6. memilih model hanya berdasarkan validation WMAPE;
7. memisahkan model prediktif ex-ante dari model intervensi conditional/ex-post;
8. mengekspor output yang konsisten untuk Bab IV dan Power BI.




## 1. Persiapan lingkungan


In [ ]:
from __future__ import annotations

import json
import logging
import platform
import shutil
import sys
import warnings
from dataclasses import dataclass
from itertools import product
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

try:
    import holidays
except ImportError as exc:
    raise ImportError(
        "Package 'holidays' belum terpasang. Jalankan: python -m pip install holidays"
    ) from exc

try:
    import prophet
    from prophet import Prophet
    from prophet.utilities import regressor_coefficients
except ImportError as exc:
    raise ImportError(
        "Package 'prophet' belum terpasang. Jalankan: python -m pip install prophet"
    ) from exc

logging.getLogger("prophet").setLevel(logging.ERROR)
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
np.random.seed(42)

if sys.version_info >= (3, 13):
    warnings.warn(
        "Python 3.13 terdeteksi. Jika Prophet gagal pada proses Stan, jalankan notebook pada Python 3.10–3.12 atau Google Colab.",
        RuntimeWarning,
    )

print("Python :", sys.version.split()[0])
print("Prophet:", getattr(prophet, "__version__", "unknown"))
print("Pandas :", pd.__version__)
print("NumPy  :", np.__version__)


## 2. Konfigurasi penelitian dan sumber regressor

Desain evaluasi diselaraskan dengan data empat tahun:

- **Training:** tahun sumber 2022–2024.
- **Validation:** tahun sumber 2025 minggu 1–12. Pada common sample, minggu pertama tidak dipakai karena tidak memiliki lag satu minggu yang valid.
- **Testing:** tahun sumber 2025 minggu 13–52.
- Dengan desain tersebut, sekitar 20% observasi terakhir berfungsi sebagai test set.
- **WMAPE** digunakan sebagai metrik utama pemilihan model.
- Yearly seasonality diuji melalui `fourier_order = 0, 5, 8`; nilai 0 berarti seasonality tahunan dinonaktifkan.
- Model intervensi tidak eligible untuk pemilihan model prediktif karena dummy ditentukan setelah shock teridentifikasi.


In [ ]:
# -----------------------------
# File, wilayah, dan folder
# -----------------------------
EXCEL_FILENAME = "clear format.xlsx"
SHEET_NAME = "Sheet1"
OUTPUT_DIR_NAME = "prophet_sultra_outputs"
CLEAN_OUTPUT_DIR = True
TARGET_UNIT = "thousand sticks"
STUDY_AREA = "Provinsi Sulawesi Tenggara"

# Data final yang diwajibkan.
REQUIRED_SOURCE_YEARS = [2022, 2023, 2024, 2025]
EXPECTED_SOURCE_WEEKS = list(range(1, 53))

# -----------------------------
# Desain evaluasi
# -----------------------------
VALIDATION_SOURCE_YEAR = 2025
VALIDATION_START_SOURCE_WEEK = 1
TEST_START_SOURCE_WEEK = 13

INTERVENTION_START_DATE = pd.Timestamp("2025-08-18")
INTERVENTION_END_DATE = pd.Timestamp("2025-09-22")

USE_LOG1P_TARGET = True
PRIMARY_SELECTION_METRIC = "WMAPE(%)"
TIE_TOLERANCE_WMAPE = 0.10  # poin persentase
INTERVAL_WIDTH = 0.80
RUN_EXTENDED_ABLATION = True

# -----------------------------
# Regressor proposal
# Dipetakan menggunakan source_year dari file Excel.
# -----------------------------
# UMP Sulawesi Tenggara.
UMP_SULTRA_BY_SOURCE_YEAR = {
    2022: 2_576_017.00,
    2023: 2_758_985.00,
    2024: 2_885_964.04,
    2025: 3_073_551.70,
}

# Perubahan rata-rata kebijakan tarif CHT nasional per tahun.
# Ini bukan tarif per batang dan bukan HJE produk tertentu.
CHT_POLICY_CHANGE_PCT_BY_SOURCE_YEAR = {
    2022: 12.0,
    2023: 10.0,
    2024: 10.0,
    2025: 0.0,
}

# Indeks kumulatif level kebijakan CHT, basis 2021 = 100.
CHT_POLICY_INDEX_BY_SOURCE_YEAR: dict[int, float] = {}
_cht_index_level = 100.0
for _year in sorted(CHT_POLICY_CHANGE_PCT_BY_SOURCE_YEAR):
    _cht_index_level *= 1.0 + CHT_POLICY_CHANGE_PCT_BY_SOURCE_YEAR[_year] / 100.0
    CHT_POLICY_INDEX_BY_SOURCE_YEAR[_year] = round(_cht_index_level, 6)

UMP_BPS_SOURCE_URL = (
    "https://sultra.bps.go.id/en/statistics-table/1/NDc4MyMx/upah-minimum-provinsi-u"
)
UMP_2025_SOURCE_URL = "https://ppid.sultraprov.go.id/front/dokumen/detail/300369228"
CHT_2022_SOURCE_URL = "https://komwasjak.kemenkeu.go.id/in/post/tarif-cukai-hasil-tembakau"
CHT_2023_2024_SOURCE_URL = (
    "https://www.kemenkeu.go.id/informasi-publik/publikasi/berita-utama/"
    "Sesuaikan-Tarif-CHT-Tahun-2023-dan-2024"
)
CHT_2025_SOURCE_URL = (
    "https://media.kemenkeu.go.id/getmedia/b6026682-cd60-48f9-9183-f03d51df9479/"
    "Publikasi-Web-Konpers-APBN-Kita-%28Maret-2025%29.pdf?ext=.pdf"
)

_regressor_rows: list[dict[str, Any]] = []
for _year, _value in UMP_SULTRA_BY_SOURCE_YEAR.items():
    _regressor_rows.append(
        {
            "variable": "UMP_SULTRA",
            "source_year": _year,
            "value": _value,
            "unit": "rupiah per bulan",
            "scope": STUDY_AREA,
            "definition": "Upah Minimum Provinsi Sulawesi Tenggara",
            "source_url": UMP_BPS_SOURCE_URL if _year <= 2024 else UMP_2025_SOURCE_URL,
        }
    )

for _year, _index_value in CHT_POLICY_INDEX_BY_SOURCE_YEAR.items():
    if _year == 2022:
        _source_url = CHT_2022_SOURCE_URL
    elif _year in (2023, 2024):
        _source_url = CHT_2023_2024_SOURCE_URL
    else:
        _source_url = CHT_2025_SOURCE_URL

    _regressor_rows.append(
        {
            "variable": "CHT_POLICY_INDEX",
            "source_year": _year,
            "value": _index_value,
            "annual_policy_change_pct": CHT_POLICY_CHANGE_PCT_BY_SOURCE_YEAR[_year],
            "unit": "indeks, basis 2021 = 100",
            "scope": "Nasional",
            "definition": (
                "Indeks kumulatif proxy level kebijakan tarif CHT sigaret; "
                "bukan tarif per batang dan bukan HJE"
            ),
            "source_url": _source_url,
        }
    )

REGRESSOR_SOURCE_TABLE = pd.DataFrame(_regressor_rows)

# Grid parameter aktif. Yearly fourier order = 0 berarti seasonality tahunan tidak digunakan.
# Grid berisi 18 kombinasi per model agar tetap cukup komprehensif tetapi masih realistis dijalankan.
PARAMETER_GRID = {
    "changepoint_prior_scale": [0.01, 0.05, 0.10],
    "n_changepoints": [10, 20],
    "changepoint_range": [0.80],
    "yearly_fourier_order": [0, 5, 8],
}


def resolve_input_file(filename: str) -> Path:
    candidates = [
        Path(filename),
        Path.cwd() / filename,
        Path("/mnt/data") / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        f"File {filename!r} tidak ditemukan. Letakkan file di folder yang sama dengan notebook atau /mnt/data."
    )


DATA_FILE = resolve_input_file(EXCEL_FILENAME)
OUTPUT_DIR = DATA_FILE.parent / OUTPUT_DIR_NAME

if CLEAN_OUTPUT_DIR and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Wilayah:", STUDY_AREA)
print("Input  :", DATA_FILE)
print("Output :", OUTPUT_DIR.resolve())
print("Tahun wajib:", REQUIRED_SOURCE_YEARS)
print("UMP mapping:", UMP_SULTRA_BY_SOURCE_YEAR)
print("CHT annual change mapping:", CHT_POLICY_CHANGE_PCT_BY_SOURCE_YEAR)
print("CHT cumulative index mapping:", CHT_POLICY_INDEX_BY_SOURCE_YEAR)
display(REGRESSOR_SOURCE_TABLE)


## 3. Definisi metrik dan struktur hasil

- **MAE/RMSE** menggunakan satuan target asli.
- **MAPE** tetap dilaporkan, tetapi sangat sensitif ketika aktual mendekati nol.
- **WMAPE** dipakai sebagai metrik utama pemilihan model.


In [ ]:
def mae_score(y_true: Any, y_pred: Any) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))


def rmse_score(y_true: Any, y_pred: Any) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mape_score(y_true: Any, y_pred: Any) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    if not mask.any():
        return float("nan")
    return float(100 * np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])))


def smape_score(y_true: Any, y_pred: Any) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denominator = np.abs(y_true) + np.abs(y_pred)
    mask = denominator != 0
    if not mask.any():
        return float("nan")
    return float(100 * np.mean(2 * np.abs(y_true[mask] - y_pred[mask]) / denominator[mask]))


def wmape_score(y_true: Any, y_pred: Any) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denominator = float(np.sum(np.abs(y_true)))
    if denominator == 0:
        return float("nan")
    return float(100 * np.sum(np.abs(y_true - y_pred)) / denominator)


@dataclass
class ForecastResult:
    model_label: str
    regressors: list[str]
    params: dict[str, Any]
    evaluation_mode: str
    mae: float
    rmse: float
    mape: float
    smape: float
    wmape: float
    forecast_df: pd.DataFrame


def summarize_metrics(y_true: Any, y_pred: Any) -> dict[str, float]:
    return {
        "MAE": mae_score(y_true, y_pred),
        "RMSE": rmse_score(y_true, y_pred),
        "MAPE(%)": mape_score(y_true, y_pred),
        "SMAPE(%)": smape_score(y_true, y_pred),
        "WMAPE(%)": wmape_score(y_true, y_pred),
    }


## 4. Pemuatan, transformasi, dan pemeriksaan data

Tahap ini melakukan:

- validasi bahwa kolom 2022–2025 tersedia;
- validasi bahwa masing-masing tahun memiliki minggu 1–52;
- transformasi format lebar ke format panjang;
- pembentukan tanggal Senin berdasarkan ISO year dan ISO week;
- pemetaan UMP Sulawesi Tenggara dan indeks kebijakan CHT;
- pembentukan dummy libur nasional;
- pembuatan `y_lag_1` hanya untuk minggu yang benar-benar berurutan;
- pembagian train, validation, dan test berdasarkan `source_year` dan `source_week`;
- penggunaan common sample agar seluruh model dibandingkan pada observasi yang sama.


In [ ]:
def iso_week_monday(source_year: int, source_week: int) -> pd.Timestamp:
    try:
        return pd.Timestamp.fromisocalendar(int(source_year), int(source_week), 1)
    except ValueError as exc:
        raise ValueError(
            f"Kombinasi ISO year-week tidak valid: year={source_year}, week={source_week}"
        ) from exc


def load_and_prepare_data(
    path: Path,
    sheet_name: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    raw = pd.read_excel(path, sheet_name=sheet_name)
    raw.columns = [str(column).strip() for column in raw.columns]

    if "week" not in raw.columns:
        raise ValueError("Kolom 'week' tidak ditemukan.")

    year_columns = sorted(
        [column for column in raw.columns if str(column).isdigit()],
        key=int,
    )
    found_years = [int(column) for column in year_columns]
    missing_years = sorted(set(REQUIRED_SOURCE_YEARS) - set(found_years))
    if missing_years:
        raise ValueError(
            "Dataset final harus memuat kolom tahun 2022–2025. "
            f"Kolom tahun yang belum tersedia: {missing_years}. "
            f"Kolom yang ditemukan: {found_years}"
        )

    # Hanya empat tahun penelitian yang digunakan, meskipun file memiliki kolom tahun lain.
    selected_year_columns = [str(year) for year in REQUIRED_SOURCE_YEARS]

    long_df = raw.melt(
        id_vars=["week"],
        value_vars=selected_year_columns,
        var_name="source_year",
        value_name="y",
    )
    long_df["source_year"] = pd.to_numeric(long_df["source_year"], errors="coerce")
    long_df["source_week"] = pd.to_numeric(long_df["week"], errors="coerce")
    long_df["y"] = pd.to_numeric(long_df["y"], errors="coerce")
    long_df = long_df.dropna(subset=["source_year", "source_week", "y"]).copy()
    long_df["source_year"] = long_df["source_year"].astype(int)
    long_df["source_week"] = long_df["source_week"].astype(int)

    duplicate_mask = long_df.duplicated(["source_year", "source_week"], keep=False)
    if duplicate_mask.any():
        duplicates = long_df.loc[duplicate_mask, ["source_year", "source_week"]]
        raise ValueError(f"Ditemukan observasi tahun-minggu duplikat:\n{duplicates}")

    # Wajib 52 minggu per tahun sesuai dataset final penelitian.
    completeness_rows: list[dict[str, Any]] = []
    completeness_errors: list[str] = []
    for year in REQUIRED_SOURCE_YEARS:
        observed_weeks = sorted(
            long_df.loc[long_df["source_year"] == year, "source_week"].unique().tolist()
        )
        missing_weeks = sorted(set(EXPECTED_SOURCE_WEEKS) - set(observed_weeks))
        extra_weeks = sorted(set(observed_weeks) - set(EXPECTED_SOURCE_WEEKS))
        completeness_rows.append(
            {
                "source_year": year,
                "observations": len(observed_weeks),
                "min_week": min(observed_weeks) if observed_weeks else np.nan,
                "max_week": max(observed_weeks) if observed_weeks else np.nan,
                "missing_weeks": ", ".join(map(str, missing_weeks)) or "-",
                "extra_weeks": ", ".join(map(str, extra_weeks)) or "-",
                "complete_52_weeks": not missing_weeks and not extra_weeks,
            }
        )
        if missing_weeks or extra_weeks:
            completeness_errors.append(
                f"{year}: missing={missing_weeks or '-'}, extra={extra_weeks or '-'}"
            )

    completeness_table = pd.DataFrame(completeness_rows)
    if completeness_errors:
        display(completeness_table)
        raise ValueError(
            "Data setiap tahun harus berisi minggu 1–52. Masalah: "
            + "; ".join(completeness_errors)
        )

    long_df["ds"] = [
        iso_week_monday(year, week)
        for year, week in zip(long_df["source_year"], long_df["source_week"])
    ]
    long_df = long_df.sort_values(["source_year", "source_week"]).reset_index(drop=True)

    # Regressor libur: 1 jika terdapat hari libur nasional pada minggu tersebut.
    holiday_years = range(long_df["ds"].dt.year.min(), long_df["ds"].dt.year.max() + 2)
    indonesia_holidays = holidays.Indonesia(years=holiday_years)
    holiday_dates = {pd.Timestamp(day).normalize() for day in indonesia_holidays.keys()}

    long_df["Libur"] = [
        int(
            any(
                (week_start + pd.Timedelta(days=offset)).normalize() in holiday_dates
                for offset in range(7)
            )
        )
        for week_start in long_df["ds"]
    ]

    # Regressor proposal menggunakan tahun sumber asli.
    long_df["UMP_SULTRA"] = long_df["source_year"].map(UMP_SULTRA_BY_SOURCE_YEAR)
    long_df["CHT_POLICY_INDEX"] = long_df["source_year"].map(
        CHT_POLICY_INDEX_BY_SOURCE_YEAR
    )

    required_regressors = ["UMP_SULTRA", "CHT_POLICY_INDEX"]
    if long_df[required_regressors].isna().any().any():
        missing_mapping_years = sorted(
            set(
                long_df.loc[
                    long_df[required_regressors].isna().any(axis=1),
                    "source_year",
                ]
            )
        )
        raise ValueError(
            "Nilai UMP Sultra atau indeks kebijakan CHT belum tersedia "
            f"untuk tahun: {missing_mapping_years}"
        )

    base_ump = float(UMP_SULTRA_BY_SOURCE_YEAR[min(REQUIRED_SOURCE_YEARS)])
    long_df["UMP_IDX"] = long_df["UMP_SULTRA"] / base_ump
    long_df["CHT_POLICY_INDEX"] = long_df["CHT_POLICY_INDEX"].astype(float)

    # Lag valid hanya dalam source_year yang sama dan selisih tepat satu minggu.
    long_df["previous_source_week"] = long_df.groupby("source_year")["source_week"].shift(1)
    long_df["previous_ds"] = long_df.groupby("source_year")["ds"].shift(1)
    long_df["y_lag_1"] = long_df.groupby("source_year")["y"].shift(1)

    is_consecutive = (
        (long_df["source_week"] - long_df["previous_source_week"] == 1)
        & ((long_df["ds"] - long_df["previous_ds"]).dt.days == 7)
    )
    long_df["lag_is_valid"] = is_consecutive.astype(int)
    long_df.loc[~is_consecutive, "y_lag_1"] = np.nan

    # Satu dummy intervensi, khusus analisis conditional/ex-post.
    long_df["INTERVENTION_2025"] = (
        long_df["ds"].between(
            INTERVENTION_START_DATE,
            INTERVENTION_END_DATE,
            inclusive="both",
        )
    ).astype(int)

    long_df["is_small_actual"] = (long_df["y"] < 0.5).astype(int)

    # Split menggunakan source year/week agar tidak terpengaruh tanggal ISO week 1.
    train_condition = long_df["source_year"] < VALIDATION_SOURCE_YEAR
    validation_condition = (
        (long_df["source_year"] == VALIDATION_SOURCE_YEAR)
        & (long_df["source_week"] >= VALIDATION_START_SOURCE_WEEK)
        & (long_df["source_week"] < TEST_START_SOURCE_WEEK)
    )
    test_condition = (
        (long_df["source_year"] == VALIDATION_SOURCE_YEAR)
        & (long_df["source_week"] >= TEST_START_SOURCE_WEEK)
    )

    long_df["split"] = np.select(
        [train_condition, validation_condition, test_condition],
        ["train", "validation", "test"],
        default="unassigned",
    )

    # Common sample: minggu pertama setiap tahun tidak digunakan karena lag belum tersedia.
    analysis_df = long_df.dropna(subset=["y_lag_1"]).copy().reset_index(drop=True)

    if (analysis_df["split"] == "unassigned").any():
        unassigned = analysis_df.loc[
            analysis_df["split"] == "unassigned",
            ["source_year", "source_week", "ds"],
        ]
        raise ValueError(f"Terdapat observasi yang tidak masuk split:\n{unassigned}")

    return long_df, analysis_df, completeness_table


raw_long_df, df, year_completeness = load_and_prepare_data(DATA_FILE, SHEET_NAME)
train_df = df[df["split"] == "train"].copy()
valid_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()
train_valid_df = df[df["split"].isin(["train", "validation"])].copy()

if min(len(train_df), len(valid_df), len(test_df)) == 0:
    raise ValueError("Salah satu split kosong. Periksa konfigurasi source year/week.")

split_summary = (
    df.groupby("split", sort=False)
    .agg(
        observations=("y", "size"),
        start_source_year=("source_year", "min"),
        start_source_week=("source_week", "min"),
        end_source_year=("source_year", "max"),
        end_source_week=("source_week", "max"),
        start_date=("ds", "min"),
        end_date=("ds", "max"),
        intervention_weeks=("INTERVENTION_2025", "sum"),
        small_actual_weeks=("is_small_actual", "sum"),
    )
    .reset_index()
)

quality_checks = pd.DataFrame(
    [
        {"check": "study area", "value": STUDY_AREA, "status": "INFO"},
        {"check": "source years", "value": ", ".join(map(str, sorted(df["source_year"].unique()))), "status": "PASS"},
        {"check": "raw observations", "value": len(raw_long_df), "status": "PASS" if len(raw_long_df) == 208 else "CHECK"},
        {"check": "common analysis observations", "value": len(df), "status": "PASS" if len(df) == 204 else "CHECK"},
        {"check": "duplicate source year-week", "value": int(raw_long_df.duplicated(["source_year", "source_week"]).sum()), "status": "PASS"},
        {"check": "missing target", "value": int(raw_long_df["y"].isna().sum()), "status": "PASS"},
        {"check": "invalid lag retained", "value": int(((df["lag_is_valid"] == 0) | df["y_lag_1"].isna()).sum()), "status": "PASS"},
        {"check": "intervention observations in train", "value": int(train_df["INTERVENTION_2025"].sum()), "status": "PASS"},
        {"check": "intervention observations in validation", "value": int(valid_df["INTERVENTION_2025"].sum()), "status": "PASS"},
        {"check": "intervention observations in test", "value": int(test_df["INTERVENTION_2025"].sum()), "status": "INFO"},
    ]
)

print(f"Wilayah               : {STUDY_AREA}")
print(f"Rentang common sample : {df['ds'].min().date()} s.d. {df['ds'].max().date()}")
print(f"Observasi mentah      : {len(raw_long_df)}")
print(f"Observasi common sample: {len(df)}")
display(year_completeness)
display(split_summary)
display(quality_checks)
display(
    df[
        [
            "source_year",
            "source_week",
            "ds",
            "y",
            "y_lag_1",
            "Libur",
            "UMP_SULTRA",
            "UMP_IDX",
            "CHT_POLICY_INDEX",
            "INTERVENTION_2025",
            "split",
        ]
    ].head(12)
)


## 5. Spesifikasi model

`baseline_only` tetap dipertahankan sesuai struktur penelitian, tetapi bukan baseline Prophet murni karena menggunakan `Libur` dan `y_lag_1`. Notebook menambahkan `prophet_only`, `naive_lag1`, `ump_only`, dan `cht_policy_only` agar kontribusi masing-masing regressor dapat diuji secara lebih transparan.


In [ ]:
MODEL_SPECS: dict[str, dict[str, Any]] = {
    "prophet_only": {
        "regressors": [],
        "group": "benchmark_prophet",
        "eligible_for_selection": True,
        "complexity_rank": 0,
        "tuning_parent": None,
        "description": "Prophet tanpa regressor; baseline Prophet murni.",
    },
    "lag_only": {
        "regressors": ["y_lag_1"],
        "group": "benchmark_prophet",
        "eligible_for_selection": True,
        "complexity_rank": 1,
        "tuning_parent": None,
        "description": "Prophet dengan kontrol historis lag satu minggu.",
    },
    "baseline_only": {
        "regressors": ["Libur", "y_lag_1"],
        "group": "research_family",
        "eligible_for_selection": True,
        "complexity_rank": 2,
        "tuning_parent": None,
        "description": "Keluarga baseline penelitian: libur dan lag satu minggu.",
    },
    "ump_only": {
        "regressors": ["UMP_IDX", "Libur", "y_lag_1"],
        "group": "regressor_diagnostic",
        "eligible_for_selection": True,
        "complexity_rank": 3,
        "tuning_parent": None,
        "description": "Mengisolasi kontribusi prediktif UMP Sulawesi Tenggara.",
    },
    "cht_policy_only": {
        "regressors": ["CHT_POLICY_INDEX", "Libur", "y_lag_1"],
        "group": "regressor_diagnostic",
        "eligible_for_selection": True,
        "complexity_rank": 3,
        "tuning_parent": None,
        "description": "Mengisolasi indeks kumulatif kebijakan tarif CHT nasional.",
    },
    "proposal_only": {
        "regressors": ["UMP_IDX", "CHT_POLICY_INDEX", "Libur", "y_lag_1"],
        "group": "research_family",
        "eligible_for_selection": True,
        "complexity_rank": 4,
        "tuning_parent": None,
        "description": "Regressor proposal: UMP Sultra, indeks kebijakan CHT, libur, dan lag.",
    },
    "proposal_plus_intervention": {
        "regressors": ["UMP_IDX", "CHT_POLICY_INDEX", "Libur", "y_lag_1", "INTERVENTION_2025"],
        "group": "conditional_ex_post",
        "eligible_for_selection": False,
        "complexity_rank": 5,
        "tuning_parent": "proposal_only",
        "description": "Model proposal dengan dummy intervensi; analisis conditional/ex-post.",
    },
    "intervention_focused": {
        "regressors": ["Libur", "y_lag_1", "INTERVENTION_2025"],
        "group": "conditional_ex_post",
        "eligible_for_selection": False,
        "complexity_rank": 3,
        "tuning_parent": "baseline_only",
        "description": "Model ringkas untuk adaptasi setelah shock teridentifikasi.",
    },
}

model_spec_table = pd.DataFrame(
    [
        {
            "model_label": label,
            "group": spec["group"],
            "eligible_for_selection": spec["eligible_for_selection"],
            "regressors": ", ".join(spec["regressors"]) or "-",
            "description": spec["description"],
        }
        for label, spec in MODEL_SPECS.items()
    ]
)
display(model_spec_table)


## 6. Fungsi Prophet dan walk-forward one-step-ahead

Setiap kandidat parameter menguji:

- fleksibilitas tren melalui `changepoint_prior_scale`;
- jumlah titik perubahan melalui `n_changepoints`;
- yearly seasonality melalui `yearly_fourier_order`.

`yearly_fourier_order = 0` berarti yearly seasonality dinonaktifkan. Nilai 5 dan 8 membentuk komponen Fourier tahunan dengan kompleksitas berbeda.

Pada setiap minggu validation/test, model hanya menggunakan data historis yang telah tersedia. Setelah aktual minggu tersebut diketahui, observasi dimasukkan ke history untuk prediksi minggu berikutnya.


In [ ]:
def parameter_grid() -> list[dict[str, Any]]:
    return [
        {
            "changepoint_prior_scale": cps,
            "n_changepoints": ncp,
            "changepoint_range": cpr,
            "yearly_fourier_order": yearly_order,
        }
        for cps, ncp, cpr, yearly_order in product(
            PARAMETER_GRID["changepoint_prior_scale"],
            PARAMETER_GRID["n_changepoints"],
            PARAMETER_GRID["changepoint_range"],
            PARAMETER_GRID["yearly_fourier_order"],
        )
    ]


def build_model(
    regressors: list[str],
    params: dict[str, Any],
    uncertainty_samples: int = 0,
) -> Prophet:
    model = Prophet(
        growth="linear",
        changepoint_prior_scale=params["changepoint_prior_scale"],
        seasonality_mode="additive",
        weekly_seasonality=False,
        yearly_seasonality=False,
        daily_seasonality=False,
        n_changepoints=params["n_changepoints"],
        changepoint_range=params["changepoint_range"],
        interval_width=INTERVAL_WIDTH,
        uncertainty_samples=uncertainty_samples,
    )

    yearly_order = int(params.get("yearly_fourier_order", 0))
    if yearly_order > 0:
        model.add_seasonality(
            name="yearly",
            period=365.25,
            fourier_order=yearly_order,
            prior_scale=10.0,
            mode="additive",
        )

    for regressor in regressors:
        model.add_regressor(regressor, standardize="auto")
    return model


def prepare_train_frame(data: pd.DataFrame, regressors: list[str]) -> pd.DataFrame:
    frame = data[["ds", "y"] + regressors].dropna().copy()
    if USE_LOG1P_TARGET:
        if (frame["y"] < 0).any():
            raise ValueError("Target mengandung nilai negatif; log1p tidak dapat digunakan.")
        frame["y"] = np.log1p(frame["y"])
    return frame


def inverse_target(values: Any) -> np.ndarray:
    array = np.asarray(values, dtype=float)
    if USE_LOG1P_TARGET:
        array = np.expm1(array)
    return np.maximum(array, 0.0)


def fit_prophet(model: Prophet, frame: pd.DataFrame) -> Prophet:
    """Fit dengan LBFGS; beri pesan yang jelas jika environment Stan tidak kompatibel."""
    try:
        return model.fit(frame, algorithm="LBFGS")
    except TypeError:
        # Kompatibilitas untuk versi Prophet yang tidak menerima argumen algorithm.
        return model.fit(frame)
    except RuntimeError as exc:
        raise RuntimeError(
            "Prophet/Stan gagal melakukan optimasi. Jalankan notebook pada Python 3.10–3.12 "
            "atau Google Colab, lalu lakukan Restart Runtime dan Run All."
        ) from exc


def walk_forward_forecast(
    history_df: pd.DataFrame,
    future_df: pd.DataFrame,
    regressors: list[str],
    params: dict[str, Any],
    model_label: str,
    evaluation_mode: str,
    uncertainty_samples: int = 0,
) -> ForecastResult:
    history = history_df.copy().reset_index(drop=True)
    future = future_df.copy().reset_index(drop=True)
    predictions: list[dict[str, Any]] = []

    for _, future_row in future.iterrows():
        model = build_model(regressors, params, uncertainty_samples=uncertainty_samples)
        model = fit_prophet(model, prepare_train_frame(history, regressors))

        future_input = pd.DataFrame([future_row[["ds"] + regressors]])
        forecast = model.predict(future_input)

        yhat = float(inverse_target(forecast["yhat"])[0])
        yhat_lower = (
            float(inverse_target(forecast["yhat_lower"])[0])
            if "yhat_lower" in forecast.columns
            else np.nan
        )
        yhat_upper = (
            float(inverse_target(forecast["yhat_upper"])[0])
            if "yhat_upper" in forecast.columns
            else np.nan
        )

        actual = float(future_row["y"])
        residual = actual - yhat
        predictions.append(
            {
                "ds": future_row["ds"],
                "source_year": int(future_row["source_year"]),
                "source_week": int(future_row["source_week"]),
                "y_actual": actual,
                "yhat": yhat,
                "yhat_lower": yhat_lower,
                "yhat_upper": yhat_upper,
                "residual": residual,
                "abs_error": abs(residual),
                "ape_pct": np.nan if actual == 0 else 100 * abs(residual) / abs(actual),
                "model_label": model_label,
                "evaluation_mode": evaluation_mode,
                "is_intervention": int(future_row["INTERVENTION_2025"]),
                "is_small_actual": int(future_row["is_small_actual"]),
            }
        )

        # Aktual minggu berjalan telah tersedia sebelum forecast minggu berikutnya.
        history = pd.concat([history, pd.DataFrame([future_row])], ignore_index=True)

    forecast_df = pd.DataFrame(predictions)
    metrics = summarize_metrics(forecast_df["y_actual"], forecast_df["yhat"])

    return ForecastResult(
        model_label=model_label,
        regressors=regressors,
        params=params,
        evaluation_mode=evaluation_mode,
        mae=metrics["MAE"],
        rmse=metrics["RMSE"],
        mape=metrics["MAPE(%)"],
        smape=metrics["SMAPE(%)"],
        wmape=metrics["WMAPE(%)"],
        forecast_df=forecast_df,
    )


def naive_lag_forecast(future_df: pd.DataFrame, evaluation_mode: str) -> ForecastResult:
    forecast_df = future_df[
        ["ds", "source_year", "source_week", "y", "y_lag_1", "INTERVENTION_2025", "is_small_actual"]
    ].copy()
    forecast_df = forecast_df.rename(columns={"y": "y_actual", "y_lag_1": "yhat"})
    forecast_df["yhat_lower"] = np.nan
    forecast_df["yhat_upper"] = np.nan
    forecast_df["residual"] = forecast_df["y_actual"] - forecast_df["yhat"]
    forecast_df["abs_error"] = forecast_df["residual"].abs()
    forecast_df["ape_pct"] = np.where(
        forecast_df["y_actual"] == 0,
        np.nan,
        100 * forecast_df["abs_error"] / forecast_df["y_actual"].abs(),
    )
    forecast_df["model_label"] = "naive_lag1"
    forecast_df["evaluation_mode"] = evaluation_mode
    forecast_df["is_intervention"] = forecast_df.pop("INTERVENTION_2025")

    metrics = summarize_metrics(forecast_df["y_actual"], forecast_df["yhat"])
    return ForecastResult(
        model_label="naive_lag1",
        regressors=["y_lag_1"],
        params={},
        evaluation_mode=evaluation_mode,
        mae=metrics["MAE"],
        rmse=metrics["RMSE"],
        mape=metrics["MAPE(%)"],
        smape=metrics["SMAPE(%)"],
        wmape=metrics["WMAPE(%)"],
        forecast_df=forecast_df,
    )


def tune_model_family(
    model_label: str,
    regressors: list[str],
    train_data: pd.DataFrame,
    validation_data: pd.DataFrame,
) -> tuple[pd.DataFrame, ForecastResult]:
    rows: list[dict[str, Any]] = []
    successful_results: list[ForecastResult] = []

    for params in parameter_grid():
        try:
            result = walk_forward_forecast(
                history_df=train_data,
                future_df=validation_data,
                regressors=regressors,
                params=params,
                model_label=model_label,
                evaluation_mode="validation_ex_ante",
                uncertainty_samples=0,
            )
            successful_results.append(result)
            rows.append(
                {
                    "model_label": model_label,
                    "regressors": ", ".join(regressors) or "-",
                    **params,
                    "MAE": result.mae,
                    "RMSE": result.rmse,
                    "MAPE(%)": result.mape,
                    "SMAPE(%)": result.smape,
                    "WMAPE(%)": result.wmape,
                    "status": "success",
                    "error": "",
                }
            )
        except Exception as exc:
            rows.append(
                {
                    "model_label": model_label,
                    "regressors": ", ".join(regressors) or "-",
                    **params,
                    "MAE": np.nan,
                    "RMSE": np.nan,
                    "MAPE(%)": np.nan,
                    "SMAPE(%)": np.nan,
                    "WMAPE(%)": np.nan,
                    "status": "failed",
                    "error": str(exc),
                }
            )

    if not successful_results:
        failed = pd.DataFrame(rows)
        display(failed.tail())
        raise RuntimeError(f"Semua kombinasi parameter gagal untuk {model_label}.")

    best_result = min(
        successful_results,
        key=lambda result: (result.wmape, result.mape, result.rmse, result.mae),
    )
    grid_df = pd.DataFrame(rows).sort_values(
        ["WMAPE(%)", "MAPE(%)", "RMSE", "MAE"],
        na_position="last",
    )
    return grid_df, best_result


## 7. Tuning dan evaluasi validation set

Seluruh model **ex-ante** dituning secara mandiri dengan grid yang sama:

- `prophet_only`
- `lag_only`
- `baseline_only`
- `ump_only`
- `cht_policy_only`
- `proposal_only`

Model intervensi mewarisi parameter model induk karena dummy intervensi bernilai nol pada training dan validation. Notebook juga memeriksa bahwa seluruh model memiliki parameter sebelum tahap testing, sehingga `KeyError` tidak terulang.


In [ ]:
tuned_results: dict[str, ForecastResult] = {}
validation_grid_frames: list[pd.DataFrame] = []

models_to_tune = [
    "prophet_only",
    "lag_only",
    "baseline_only",
    "ump_only",
    "cht_policy_only",
    "proposal_only",
]

for model_label in models_to_tune:
    spec = MODEL_SPECS[model_label]
    print(f"Tuning {model_label} ...")
    grid_df, best_result = tune_model_family(
        model_label=model_label,
        regressors=spec["regressors"],
        train_data=train_df,
        validation_data=valid_df,
    )
    validation_grid_frames.append(grid_df)
    tuned_results[model_label] = best_result
    print(
        f"  best validation WMAPE={best_result.wmape:.4f}% | "
        f"MAPE={best_result.mape:.4f}% | params={best_result.params}"
    )

# Model conditional mewarisi parameter model induk.
for model_label in ["proposal_plus_intervention", "intervention_focused"]:
    spec = MODEL_SPECS[model_label]
    parent_label = spec["tuning_parent"]
    parent_result = tuned_results[parent_label]
    tuned_results[model_label] = ForecastResult(
        model_label=model_label,
        regressors=spec["regressors"],
        params=parent_result.params.copy(),
        evaluation_mode="validation_dummy_inactive",
        mae=parent_result.mae,
        rmse=parent_result.rmse,
        mape=parent_result.mape,
        smape=parent_result.smape,
        wmape=parent_result.wmape,
        forecast_df=parent_result.forecast_df.copy(),
    )

# Cegah KeyError pada tahap testing.
missing_models = [
    model_label
    for model_label in MODEL_SPECS
    if model_label not in tuned_results
]
if missing_models:
    raise RuntimeError(
        "Model berikut belum memiliki hasil tuning atau parameter induk: "
        f"{missing_models}"
    )
print("Semua model sudah memiliki parameter untuk tahap pengujian.")

naive_validation = naive_lag_forecast(valid_df, evaluation_mode="validation_ex_ante")

validation_rows = [
    {
        "model_label": "naive_lag1",
        "model_group": "benchmark_naive",
        "eligible_for_prophet_selection": False,
        "complexity_rank": -1,
        "regressors": "y_lag_1",
        "parameter_source": "not_applicable",
        "MAE": naive_validation.mae,
        "RMSE": naive_validation.rmse,
        "MAPE(%)": naive_validation.mape,
        "SMAPE(%)": naive_validation.smape,
        "WMAPE(%)": naive_validation.wmape,
        "params": "{}",
        "validation_note": "Naive benchmark; tidak termasuk pemilihan final Prophet.",
    }
]

for model_label, result in tuned_results.items():
    spec = MODEL_SPECS[model_label]
    parent = spec["tuning_parent"]
    validation_rows.append(
        {
            "model_label": model_label,
            "model_group": spec["group"],
            "eligible_for_prophet_selection": spec["eligible_for_selection"],
            "complexity_rank": spec["complexity_rank"],
            "regressors": ", ".join(spec["regressors"]) or "-",
            "parameter_source": parent or model_label,
            "MAE": result.mae,
            "RMSE": result.rmse,
            "MAPE(%)": result.mape,
            "SMAPE(%)": result.smape,
            "WMAPE(%)": result.wmape,
            "yearly_fourier_order": result.params.get("yearly_fourier_order", np.nan),
            "params": json.dumps(result.params, sort_keys=True),
            "validation_note": (
                "Dummy intervensi tidak aktif pada train/validation; metrik sama dengan model induk."
                if spec["group"] == "conditional_ex_post"
                else "Eligible untuk pemilihan model Prophet berdasarkan validation set."
            ),
        }
    )

validation_comparison = pd.DataFrame(validation_rows).sort_values(
    ["WMAPE(%)", "MAPE(%)", "RMSE", "complexity_rank"]
).reset_index(drop=True)
validation_grid_all = pd.concat(validation_grid_frames, ignore_index=True)

display(validation_comparison)


## 8. Pemilihan model dengan aturan tie dan parsimony

Model dipilih **hanya dari validation set** menggunakan WMAPE. Jika beberapa model berada dalam toleransi `0,10` poin WMAPE, model yang lebih sederhana diprioritaskan.

Hasil test tidak digunakan untuk mengganti model final. Model conditional/ex-post juga tidak eligible untuk pemilihan prediktif.


In [ ]:
eligible_validation = validation_comparison[
    validation_comparison["eligible_for_prophet_selection"]
].copy()

best_wmape = eligible_validation["WMAPE(%)"].min()
tied_candidates = eligible_validation[
    eligible_validation["WMAPE(%)"] <= best_wmape + TIE_TOLERANCE_WMAPE
].copy()

tied_candidates = tied_candidates.sort_values(
    ["complexity_rank", "WMAPE(%)", "MAPE(%)", "RMSE"]
)
selected_predictive_model = tied_candidates.iloc[0]["model_label"]
selected_validation_result = tuned_results[selected_predictive_model]

naive_wmape = float(
    validation_comparison.loc[
        validation_comparison["model_label"] == "naive_lag1", "WMAPE(%)"
    ].iloc[0]
)
selected_beats_naive_validation = selected_validation_result.wmape < naive_wmape

selection_summary = pd.DataFrame(
    [
        {
            "selected_predictive_model": selected_predictive_model,
            "primary_metric": PRIMARY_SELECTION_METRIC,
            "selected_validation_WMAPE(%)": selected_validation_result.wmape,
            "naive_validation_WMAPE(%)": naive_wmape,
            "beats_naive_on_validation": selected_beats_naive_validation,
            "tie_tolerance_WMAPE_point": TIE_TOLERANCE_WMAPE,
            "number_of_tied_candidates": len(tied_candidates),
            "selection_basis": "Validation WMAPE + parsimony; test set tidak digunakan.",
        }
    ]
)

display(Markdown(f"### Model prediktif terpilih: `{selected_predictive_model}`"))
display(selection_summary)
display(tied_candidates[["model_label", "WMAPE(%)", "MAPE(%)", "RMSE", "complexity_rank"]])


## 9. Evaluasi final pada test set

- Model prediktif final tetap model yang telah dipilih dari validation set.
- Seluruh model ex-ante tetap dievaluasi pada test untuk pembahasan komparatif.
- Model intervensi dilaporkan terpisah sebagai conditional/ex-post.
- Parameter yearly seasonality yang terpilih ikut tersimpan dalam seluruh output.


In [ ]:
test_results: dict[str, ForecastResult] = {
    "naive_lag1": naive_lag_forecast(test_df, evaluation_mode="test_ex_ante")
}

for model_label, spec in MODEL_SPECS.items():
    evaluation_mode = (
        "test_conditional_ex_post"
        if spec["group"] == "conditional_ex_post"
        else "test_ex_ante"
    )
    tuned = tuned_results[model_label]
    print(f"Testing {model_label} ...")
    test_results[model_label] = walk_forward_forecast(
        history_df=train_valid_df,
        future_df=test_df,
        regressors=spec["regressors"],
        params=tuned.params,
        model_label=model_label,
        evaluation_mode=evaluation_mode,
        uncertainty_samples=0,
    )

# Rerun model prediktif terpilih dengan interval prediksi untuk visualisasi final.
selected_predictive_result = walk_forward_forecast(
    history_df=train_valid_df,
    future_df=test_df,
    regressors=MODEL_SPECS[selected_predictive_model]["regressors"],
    params=tuned_results[selected_predictive_model].params,
    model_label=selected_predictive_model,
    evaluation_mode="test_ex_ante_selected_with_interval",
    uncertainty_samples=100,
)
test_results[selected_predictive_model] = selected_predictive_result

conditional_labels = [
    label
    for label, spec in MODEL_SPECS.items()
    if spec["group"] == "conditional_ex_post"
]
conditional_ranking = sorted(
    conditional_labels,
    key=lambda label: test_results[label].wmape,
)
best_conditional_model_descriptive = conditional_ranking[0]

metric_rows: list[dict[str, Any]] = []
all_forecast_frames: list[pd.DataFrame] = []

for model_label, result in test_results.items():
    if model_label == "naive_lag1":
        model_group = "benchmark_naive"
        eligible = False
        regressors = ["y_lag_1"]
    else:
        spec = MODEL_SPECS[model_label]
        model_group = spec["group"]
        eligible = spec["eligible_for_selection"]
        regressors = spec["regressors"]

    metric_rows.append(
        {
            "model_label": model_label,
            "model_group": model_group,
            "evaluation_mode": result.evaluation_mode,
            "eligible_for_prophet_selection": eligible,
            "is_selected_predictive_model": int(model_label == selected_predictive_model),
            "is_best_conditional_descriptive": int(model_label == best_conditional_model_descriptive),
            "regressors": ", ".join(regressors) or "-",
            "params": json.dumps(result.params, sort_keys=True),
            "test_MAE": result.mae,
            "test_RMSE": result.rmse,
            "test_MAPE(%)": result.mape,
            "test_SMAPE(%)": result.smape,
            "test_WMAPE(%)": result.wmape,
        }
    )

    forecast = result.forecast_df.copy()
    forecast["model_group"] = model_group
    forecast["is_selected_predictive_model"] = int(model_label == selected_predictive_model)
    forecast["is_best_conditional_descriptive"] = int(model_label == best_conditional_model_descriptive)
    forecast["target_unit"] = TARGET_UNIT
    forecast["data_split"] = "test"
    all_forecast_frames.append(forecast)

all_models_test_metrics = pd.DataFrame(metric_rows).sort_values(
    ["test_WMAPE(%)", "test_MAPE(%)", "test_RMSE"]
).reset_index(drop=True)
all_models_forecast = pd.concat(all_forecast_frames, ignore_index=True).sort_values(
    ["ds", "model_label"]
).reset_index(drop=True)

selected_test_summary = pd.DataFrame(
    [
        {
            "selected_predictive_model": selected_predictive_model,
            "selected_regressors": ", ".join(MODEL_SPECS[selected_predictive_model]["regressors"]) or "-",
            "selected_params": json.dumps(tuned_results[selected_predictive_model].params, sort_keys=True),
            "selection_source": "validation set",
            "test_MAE": selected_predictive_result.mae,
            "test_RMSE": selected_predictive_result.rmse,
            "test_MAPE(%)": selected_predictive_result.mape,
            "test_SMAPE(%)": selected_predictive_result.smape,
            "test_WMAPE(%)": selected_predictive_result.wmape,
            "target_mape_below_20_achieved": bool(selected_predictive_result.mape < 20),
            "target_unit": TARGET_UNIT,
        }
    ]
)

conditional_summary = pd.DataFrame(
    [
        {
            "best_conditional_model_descriptive": best_conditional_model_descriptive,
            "selection_warning": "Ranking conditional hanya deskriptif dari test; bukan pemilihan model prediktif final.",
            "test_MAE": test_results[best_conditional_model_descriptive].mae,
            "test_RMSE": test_results[best_conditional_model_descriptive].rmse,
            "test_MAPE(%)": test_results[best_conditional_model_descriptive].mape,
            "test_SMAPE(%)": test_results[best_conditional_model_descriptive].smape,
            "test_WMAPE(%)": test_results[best_conditional_model_descriptive].wmape,
        }
    ]
)

display(all_models_test_metrics)
display(selected_test_summary)
display(conditional_summary)


## 10. Evaluasi berdasarkan rezim normal dan shock

Tabel ini memisahkan performa sebelum/dalam periode intervensi. Tujuannya untuk menunjukkan apakah dummy intervensi membantu adaptasi setelah shock mulai dikenali, tanpa mengklaim bahwa model mampu meramalkan tanggal awal shock.


In [ ]:
regime_rows: list[dict[str, Any]] = []
for model_label, result in test_results.items():
    for regime_name, regime_value in [("normal", 0), ("intervention", 1)]:
        subset = result.forecast_df[
            result.forecast_df["is_intervention"] == regime_value
        ]
        if subset.empty:
            continue
        metrics = summarize_metrics(subset["y_actual"], subset["yhat"])
        regime_rows.append(
            {
                "model_label": model_label,
                "evaluation_mode": result.evaluation_mode,
                "regime": regime_name,
                "observations": len(subset),
                **metrics,
            }
        )

metrics_by_regime = pd.DataFrame(regime_rows).sort_values(
    ["regime", "WMAPE(%)", "MAPE(%)"]
).reset_index(drop=True)
display(metrics_by_regime)


## 11. Diagnostik regressor dan hipotesis

Koefisien Prophet digunakan sebagai **indikasi arah asosiasi**, bukan bukti kausal dan bukan uji signifikansi klasik.

UMP Sulawesi Tenggara dan indeks kebijakan CHT masing-masing memiliki empat level tahunan pada periode 2022–2025. Jumlah ini lebih baik daripada dataset dua tahun, tetapi masih terbatas dan keduanya dapat berkorelasi tinggi karena sama-sama berubah pada awal tahun. Karena itu, arah dan besarnya koefisien individual harus ditafsirkan hati-hati.


In [ ]:
regressor_columns = [
    "UMP_IDX",
    "CHT_POLICY_INDEX",
    "Libur",
    "y_lag_1",
    "INTERVENTION_2025",
]


def safe_corr(frame: pd.DataFrame, first: str, second: str) -> float:
    if frame[first].nunique() <= 1 or frame[second].nunique() <= 1:
        return float("nan")
    return float(frame[[first, second]].corr().iloc[0, 1])


regressor_diagnostics = pd.DataFrame(
    [
        {
            "regressor": column,
            "unique_values": int(df[column].nunique()),
            "standard_deviation": float(df[column].std(ddof=0)),
            "correlation_with_y": safe_corr(df, column, "y"),
            "available_ex_ante": column != "INTERVENTION_2025",
            "scope": (
                STUDY_AREA
                if column == "UMP_IDX"
                else "Nasional"
                if column == "CHT_POLICY_INDEX"
                else "Temporal/data internal"
            ),
            "interpretation_warning": (
                "Dummy ditentukan setelah periode shock diidentifikasi; hanya untuk conditional analysis."
                if column == "INTERVENTION_2025"
                else "Proxy perubahan kebijakan nasional, bukan tarif per batang atau HJE."
                if column == "CHT_POLICY_INDEX"
                else ""
            ),
        }
        for column in regressor_columns
    ]
)

regressor_correlation_matrix = df[regressor_columns].corr()
collinearity_rows: list[dict[str, Any]] = []
for i, first in enumerate(regressor_columns):
    for second in regressor_columns[i + 1 :]:
        correlation = safe_corr(df, first, second)
        collinearity_rows.append(
            {
                "regressor_1": first,
                "regressor_2": second,
                "correlation": correlation,
                "high_collinearity_abs_ge_0_95": (
                    bool(abs(correlation) >= 0.95) if np.isfinite(correlation) else False
                ),
            }
        )
pairwise_collinearity = pd.DataFrame(collinearity_rows).sort_values(
    "correlation", key=lambda series: series.abs(), ascending=False
)

# Koefisien proposal-only, fit pada train+validation agar test tidak dipakai untuk interpretasi proposal.
proposal_regressors = MODEL_SPECS["proposal_only"]["regressors"]
proposal_model = build_model(
    proposal_regressors,
    tuned_results["proposal_only"].params,
    uncertainty_samples=100,
)
proposal_model = fit_prophet(
    proposal_model,
    prepare_train_frame(train_valid_df, proposal_regressors),
)
proposal_coefficients = regressor_coefficients(proposal_model)
proposal_coefficients["analysis_scope"] = "train_plus_validation"
proposal_coefficients["coefficient_scale"] = (
    "log1p target scale" if USE_LOG1P_TARGET else "original target scale"
)

expected_direction = {
    "CHT_POLICY_INDEX": "negative",
    "UMP_IDX": "positive",
    "Libur": "positive",
    "y_lag_1": "positive_control",
}
proposal_coefficients["expected_direction"] = proposal_coefficients["regressor"].map(expected_direction)
proposal_coefficients["actual_direction"] = np.select(
    [proposal_coefficients["coef"] > 0, proposal_coefficients["coef"] < 0],
    ["positive", "negative"],
    default="zero",
)
proposal_coefficients["direction_matches_expectation"] = np.where(
    proposal_coefficients["expected_direction"].eq("positive"),
    proposal_coefficients["actual_direction"].eq("positive"),
    np.where(
        proposal_coefficients["expected_direction"].eq("negative"),
        proposal_coefficients["actual_direction"].eq("negative"),
        np.nan,
    ),
)

ump_cht_corr = safe_corr(df, "UMP_IDX", "CHT_POLICY_INDEX")
proposal_coefficients["identifiability_warning"] = np.where(
    proposal_coefficients["regressor"].isin(["UMP_IDX", "CHT_POLICY_INDEX"])
    & np.isfinite(ump_cht_corr)
    & (abs(ump_cht_corr) >= 0.95),
    "UMP dan indeks kebijakan CHT sangat kollinear pada data empat tahun; koefisien individual tidak boleh ditafsirkan kuat.",
    "",
)
proposal_coefficients["causal_warning"] = (
    "Koefisien menunjukkan asosiasi prediktif pada sampel, bukan efek kausal."
)

# Koefisien intervensi hanya deskriptif dan memakai seluruh data agar dummy tidak konstan nol.
conditional_label_for_coef = "intervention_focused"
conditional_regressors = MODEL_SPECS[conditional_label_for_coef]["regressors"]
conditional_model = build_model(
    conditional_regressors,
    tuned_results[conditional_label_for_coef].params,
    uncertainty_samples=100,
)
conditional_model = fit_prophet(
    conditional_model,
    prepare_train_frame(df, conditional_regressors),
)
conditional_coefficients = regressor_coefficients(conditional_model)
conditional_coefficients["analysis_scope"] = "full_sample_descriptive_only"
conditional_coefficients["interpretation_warning"] = (
    "Menggunakan seluruh data dan dummy ex-post; tidak digunakan untuk pemilihan atau klaim prediksi shock."
)

display(regressor_diagnostics)
display(pairwise_collinearity.head(10))
display(proposal_coefficients)
display(conditional_coefficients)


## 12. Ablation test pada validation set

Ablation menilai kontribusi prediktif UMP Sulawesi Tenggara, indeks kebijakan CHT nasional, dan hari libur terhadap model `lag_only`. Dummy intervensi tidak dimasukkan ke ablation validation karena seluruh nilainya nol pada train/validation.


In [ ]:
ablation_specs = {
    "prophet_only": [],
    "lag_only": ["y_lag_1"],
    "libur_plus_lag": ["Libur", "y_lag_1"],
    "ump_plus_lag": ["UMP_IDX", "y_lag_1"],
    "cht_policy_plus_lag": ["CHT_POLICY_INDEX", "y_lag_1"],
    "ump_libur_lag": ["UMP_IDX", "Libur", "y_lag_1"],
    "cht_policy_libur_lag": ["CHT_POLICY_INDEX", "Libur", "y_lag_1"],
    "proposal_only": ["UMP_IDX", "CHT_POLICY_INDEX", "Libur", "y_lag_1"],
}

ablation_results: dict[str, ForecastResult] = {}
ablation_grid_frames: list[pd.DataFrame] = []

source_model_map = {
    "libur_plus_lag": "baseline_only",
    "ump_libur_lag": "ump_only",
    "cht_policy_libur_lag": "cht_policy_only",
}

for label, regressors in ablation_specs.items():
    source_label = source_model_map.get(label, label)

    if source_label in tuned_results:
        ablation_results[label] = tuned_results[source_label]
        continue

    if RUN_EXTENDED_ABLATION:
        print(f"Tuning ablation {label} ...")
        grid_df, best_result = tune_model_family(
            model_label=label,
            regressors=regressors,
            train_data=train_df,
            validation_data=valid_df,
        )
        ablation_grid_frames.append(grid_df)
        ablation_results[label] = best_result
    else:
        inherited_params = tuned_results["lag_only"].params
        ablation_results[label] = walk_forward_forecast(
            history_df=train_df,
            future_df=valid_df,
            regressors=regressors,
            params=inherited_params,
            model_label=label,
            evaluation_mode="validation_ablation_fast_mode",
            uncertainty_samples=0,
        )

lag_only_wmape = ablation_results["lag_only"].wmape
ablation_comparison = pd.DataFrame(
    [
        {
            "model_label": label,
            "regressors": ", ".join(ablation_specs[label]) or "-",
            "MAE": result.mae,
            "RMSE": result.rmse,
            "MAPE(%)": result.mape,
            "SMAPE(%)": result.smape,
            "WMAPE(%)": result.wmape,
            "delta_WMAPE_vs_lag_only": result.wmape - lag_only_wmape,
            "contribution_vs_lag_only": (
                "improves"
                if result.wmape < lag_only_wmape
                else "worsens"
                if result.wmape > lag_only_wmape
                else "equal"
            ),
            "params": json.dumps(result.params, sort_keys=True),
        }
        for label, result in ablation_results.items()
    ]
).sort_values(["WMAPE(%)", "MAPE(%)", "RMSE"]).reset_index(drop=True)

display(ablation_comparison)


## 13. Residual dan interval prediksi model terpilih


In [ ]:
selected_forecast = selected_predictive_result.forecast_df.copy()
residuals = selected_forecast["residual"]
lag1_residual_autocorrelation = (
    float(residuals.autocorr(lag=1)) if len(residuals) >= 3 else np.nan
)
interval_available = selected_forecast[["yhat_lower", "yhat_upper"]].notna().all(axis=1)
coverage = (
    float(
        (
            (selected_forecast.loc[interval_available, "y_actual"] >= selected_forecast.loc[interval_available, "yhat_lower"])
            & (selected_forecast.loc[interval_available, "y_actual"] <= selected_forecast.loc[interval_available, "yhat_upper"])
        ).mean()
    )
    if interval_available.any()
    else np.nan
)

residual_diagnostics = pd.DataFrame(
    [
        {
            "model_label": selected_predictive_model,
            "observations": len(selected_forecast),
            "mean_residual_bias": float(residuals.mean()),
            "residual_standard_deviation": float(residuals.std(ddof=1)),
            "residual_autocorrelation_lag1": lag1_residual_autocorrelation,
            "prediction_interval_nominal_coverage": INTERVAL_WIDTH,
            "prediction_interval_empirical_coverage": coverage,
            "small_actual_observations": int(selected_forecast["is_small_actual"].sum()),
        }
    ]
)

display(residual_diagnostics)
display(selected_forecast[["ds", "source_week", "y_actual", "yhat", "yhat_lower", "yhat_upper", "residual", "ape_pct", "is_intervention"]])


## 14. Visualisasi


In [ ]:
def save_figure(filename: str) -> Path:
    path = OUTPUT_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()
    return path


# 1. Gambaran data dan split
plt.figure(figsize=(14, 5))
plt.plot(df["ds"], df["y"], marker="o", linewidth=1.5, label="Actual")
validation_start_plot = valid_df["ds"].min()
test_start_plot = test_df["ds"].min()
plt.axvline(validation_start_plot, linestyle="--", label="Validation start")
plt.axvline(test_start_plot, linestyle="--", label="Test start")
plt.axvspan(INTERVENTION_START_DATE, INTERVENTION_END_DATE, alpha=0.15, label="Intervention period")
plt.title("Data Penjualan Mingguan dan Pembagian Evaluasi")
plt.xlabel("Tanggal")
plt.ylabel(f"Penjualan ({TARGET_UNIT})")
plt.legend()
plt.grid(alpha=0.25)
save_figure("plot_01_data_and_split.png")

# 2. Perbandingan validation WMAPE
plot_validation = validation_comparison.sort_values("WMAPE(%)", ascending=True)
plt.figure(figsize=(11, 5))
plt.barh(plot_validation["model_label"], plot_validation["WMAPE(%)"])
plt.title("Perbandingan WMAPE pada Validation Set")
plt.xlabel("WMAPE (%)")
plt.ylabel("Model")
plt.grid(axis="x", alpha=0.25)
save_figure("plot_02_validation_wmape.png")

# 3. Semua forecast test
plt.figure(figsize=(14, 6))
actual_test = test_df.sort_values("ds")
plt.plot(actual_test["ds"], actual_test["y"], marker="o", linewidth=2.5, label="Actual")
for model_label in [selected_predictive_model, "proposal_plus_intervention", "intervention_focused", "naive_lag1"]:
    result = test_results[model_label].forecast_df
    plt.plot(result["ds"], result["yhat"], marker=".", linestyle="--", label=model_label)
plt.axvspan(INTERVENTION_START_DATE, INTERVENTION_END_DATE, alpha=0.15, label="Intervention period")
plt.title("Aktual vs Forecast pada Test Set")
plt.xlabel("Tanggal")
plt.ylabel(f"Penjualan ({TARGET_UNIT})")
plt.legend()
plt.grid(alpha=0.25)
save_figure("plot_03_test_forecast_comparison.png")

# 4. Model prediktif terpilih dengan interval
plt.figure(figsize=(14, 6))
plt.plot(train_valid_df["ds"], train_valid_df["y"], linewidth=1.3, label="Train + validation actual")
plt.plot(selected_forecast["ds"], selected_forecast["y_actual"], marker="o", linewidth=2, label="Test actual")
plt.plot(selected_forecast["ds"], selected_forecast["yhat"], marker="o", linestyle="--", label="Selected forecast")
if selected_forecast[["yhat_lower", "yhat_upper"]].notna().any().all():
    plt.fill_between(
        selected_forecast["ds"],
        selected_forecast["yhat_lower"],
        selected_forecast["yhat_upper"],
        alpha=0.20,
        label=f"{int(INTERVAL_WIDTH * 100)}% interval",
    )
plt.axvspan(INTERVENTION_START_DATE, INTERVENTION_END_DATE, alpha=0.15, label="Intervention period")
plt.title(f"Forecast Final: {selected_predictive_model}")
plt.xlabel("Tanggal")
plt.ylabel(f"Penjualan ({TARGET_UNIT})")
plt.legend()
plt.grid(alpha=0.25)
save_figure("plot_04_selected_model_forecast.png")

# 5. Residual test
plt.figure(figsize=(14, 4))
plt.axhline(0, linewidth=1)
plt.plot(selected_forecast["ds"], selected_forecast["residual"], marker="o")
plt.axvspan(INTERVENTION_START_DATE, INTERVENTION_END_DATE, alpha=0.15)
plt.title(f"Residual Test: {selected_predictive_model}")
plt.xlabel("Tanggal")
plt.ylabel("Aktual - Prediksi")
plt.grid(alpha=0.25)
save_figure("plot_05_selected_model_residual.png")


## 15. Ekspor output terstruktur

Output memakai awalan angka agar urutan file konsisten. File `14_powerbi_forecast.csv` dan `15_powerbi_model_metrics.csv` disiapkan khusus untuk Power BI.


In [ ]:
# Dataset, sumber regressor, dan kualitas data
REGRESSOR_SOURCE_TABLE.to_csv(OUTPUT_DIR / "00_regressor_input_sources.csv", index=False)
raw_long_df.to_csv(OUTPUT_DIR / "00_raw_long_data.csv", index=False)
df.to_csv(OUTPUT_DIR / "01_processed_common_sample.csv", index=False)
split_summary.to_csv(OUTPUT_DIR / "02_split_summary.csv", index=False)
quality_checks.to_csv(OUTPUT_DIR / "03_data_quality_checks.csv", index=False)
year_completeness.to_csv(OUTPUT_DIR / "03a_year_completeness.csv", index=False)

# Validasi dan pemilihan model
validation_grid_all.to_csv(OUTPUT_DIR / "04_validation_grid_all.csv", index=False)
validation_comparison.to_csv(OUTPUT_DIR / "05_validation_model_comparison.csv", index=False)
selection_summary.to_csv(OUTPUT_DIR / "06_model_selection_summary.csv", index=False)

# Test dan intervention analysis
all_models_test_metrics.to_csv(OUTPUT_DIR / "07_test_model_metrics.csv", index=False)
all_models_forecast.to_csv(OUTPUT_DIR / "08_all_models_test_forecast.csv", index=False)
selected_test_summary.to_csv(OUTPUT_DIR / "09_selected_predictive_model_summary.csv", index=False)
selected_predictive_result.forecast_df.to_csv(OUTPUT_DIR / "10_selected_predictive_forecast.csv", index=False)
conditional_summary.to_csv(OUTPUT_DIR / "11_conditional_intervention_summary.csv", index=False)
metrics_by_regime.to_csv(OUTPUT_DIR / "12_test_metrics_by_regime.csv", index=False)

# Regressor, ablation, dan residual
regressor_diagnostics.to_csv(OUTPUT_DIR / "13_regressor_diagnostics.csv", index=False)
regressor_correlation_matrix.to_csv(OUTPUT_DIR / "13a_regressor_correlation_matrix.csv")
pairwise_collinearity.to_csv(OUTPUT_DIR / "13b_pairwise_collinearity.csv", index=False)
proposal_coefficients.to_csv(OUTPUT_DIR / "13c_proposal_regressor_coefficients.csv", index=False)
conditional_coefficients.to_csv(OUTPUT_DIR / "13d_conditional_regressor_coefficients_descriptive.csv", index=False)
ablation_comparison.to_csv(OUTPUT_DIR / "13e_ablation_validation_comparison.csv", index=False)
if ablation_grid_frames:
    pd.concat(ablation_grid_frames, ignore_index=True).to_csv(
        OUTPUT_DIR / "13f_ablation_grid_additional.csv", index=False
    )
residual_diagnostics.to_csv(OUTPUT_DIR / "13g_selected_model_residual_diagnostics.csv", index=False)

# Output Power BI dengan nama kolom ringkas dan stabil.
powerbi_forecast = all_models_forecast.rename(
    columns={
        "ds": "Date",
        "source_year": "SourceYear",
        "source_week": "SourceWeek",
        "y_actual": "Actual",
        "yhat": "Prediction",
        "yhat_lower": "PredictionLower",
        "yhat_upper": "PredictionUpper",
        "abs_error": "AbsoluteError",
        "ape_pct": "APEPct",
        "model_label": "ModelLabel",
        "model_group": "ModelGroup",
        "evaluation_mode": "EvaluationMode",
        "is_intervention": "IsIntervention",
        "is_small_actual": "IsSmallActual",
        "is_selected_predictive_model": "IsSelectedPredictiveModel",
        "is_best_conditional_descriptive": "IsBestConditionalDescriptive",
        "target_unit": "TargetUnit",
        "data_split": "DataSplit",
    }
)
powerbi_forecast.to_csv(OUTPUT_DIR / "14_powerbi_forecast.csv", index=False)

powerbi_metrics = all_models_test_metrics.rename(
    columns={
        "model_label": "ModelLabel",
        "model_group": "ModelGroup",
        "evaluation_mode": "EvaluationMode",
        "is_selected_predictive_model": "IsSelectedPredictiveModel",
        "is_best_conditional_descriptive": "IsBestConditionalDescriptive",
        "test_MAE": "MAE",
        "test_RMSE": "RMSE",
        "test_MAPE(%)": "MAPEPct",
        "test_SMAPE(%)": "SMAPEPct",
        "test_WMAPE(%)": "WMAPEPct",
    }
)
powerbi_metrics.to_csv(OUTPUT_DIR / "15_powerbi_model_metrics.csv", index=False)

# Ringkasan manifest untuk reproduksibilitas.
manifest = {
    "input_file": str(DATA_FILE),
    "sheet_name": SHEET_NAME,
    "target_unit": TARGET_UNIT,
    "study_area": STUDY_AREA,
    "ump_sultra_by_source_year": UMP_SULTRA_BY_SOURCE_YEAR,
    "cht_policy_change_pct_by_source_year": CHT_POLICY_CHANGE_PCT_BY_SOURCE_YEAR,
    "cht_policy_index_by_source_year": CHT_POLICY_INDEX_BY_SOURCE_YEAR,
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
    "prophet_version": getattr(prophet, "__version__", "unknown"),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "required_source_years": REQUIRED_SOURCE_YEARS,
    "expected_source_weeks": EXPECTED_SOURCE_WEEKS,
    "validation_source_year": VALIDATION_SOURCE_YEAR,
    "validation_start_source_week": VALIDATION_START_SOURCE_WEEK,
    "test_start_source_week": TEST_START_SOURCE_WEEK,
    "parameter_grid": PARAMETER_GRID,
    "intervention_start_date": str(INTERVENTION_START_DATE.date()),
    "intervention_end_date": str(INTERVENTION_END_DATE.date()),
    "selected_predictive_model": selected_predictive_model,
    "best_conditional_model_descriptive": best_conditional_model_descriptive,
    "primary_selection_metric": PRIMARY_SELECTION_METRIC,
    "tie_tolerance_wmape_point": TIE_TOLERANCE_WMAPE,
    "use_log1p_target": USE_LOG1P_TARGET,
    "counts": {
        "train": len(train_df),
        "validation": len(valid_df),
        "test": len(test_df),
    },
    "methodological_warnings": [
        "Intervention dummy is ex-post and is not eligible for predictive model selection.",
        "UMP_IDX and CHT_POLICY_INDEX may still be highly collinear because both vary annually.",
        "CHT_POLICY_INDEX is a cumulative national policy index with base 2021=100, not a per-stick tariff or HJE.",
        "Evaluation is one-step-ahead walk-forward, not unrestricted multi-step forecasting.",
        "Yearly seasonality is selected empirically using validation WMAPE; it is not assumed to be active.",
        "MAPE is unstable when actual values are close to zero; WMAPE is the primary metric.",
    ],
}
with open(OUTPUT_DIR / "16_run_manifest.json", "w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=2, ensure_ascii=False)

output_readme = f"""# Output Prophet Weekly Forecast

Model prediktif terpilih: **{selected_predictive_model}**  
Model conditional terbaik secara deskriptif: **{best_conditional_model_descriptive}**

## Sumber input regressor
- `00_regressor_input_sources.csv`

## File utama untuk laporan
- `05_validation_model_comparison.csv`
- `06_model_selection_summary.csv`
- `07_test_model_metrics.csv`
- `09_selected_predictive_model_summary.csv`
- `10_selected_predictive_forecast.csv`
- `11_conditional_intervention_summary.csv`
- `12_test_metrics_by_regime.csv`
- `13c_proposal_regressor_coefficients.csv`
- `13e_ablation_validation_comparison.csv`

## File utama untuk Power BI
- `14_powerbi_forecast.csv`
- `15_powerbi_model_metrics.csv`

## Batas interpretasi
Model intervensi bersifat conditional/ex-post. Dummy intervensi tidak membuktikan kemampuan model memprediksi tanggal awal shock.
"""
(OUTPUT_DIR / "README_OUTPUTS.md").write_text(output_readme, encoding="utf-8")

print(f"Seluruh output tersimpan di: {OUTPUT_DIR.resolve()}")
print(f"Jumlah file output: {len(list(OUTPUT_DIR.iterdir()))}")


## 16. Ringkasan akhir untuk pembahasan skripsi

Gunakan ringkasan ini sebagai dasar narasi Bab IV dan V.

Batas interpretasi:

- `CHT_POLICY_INDEX` adalah indeks kumulatif kebijakan nasional, bukan tarif per batang atau HJE.
- UMP dan indeks CHT hanya memiliki empat level tahunan sehingga koefisien individual harus dibaca hati-hati.
- Yearly seasonality dipertahankan hanya jika dipilih berdasarkan validation WMAPE.
- Model intervensi hanya menjelaskan adaptasi ketika periode shock telah diketahui.
- Seluruh angka skripsi dan dashboard harus berasal dari satu kali `Restart Kernel → Run All`.


In [ ]:
selected_row = selected_test_summary.iloc[0]
conditional_row = conditional_summary.iloc[0]

executive_summary = pd.DataFrame(
    [
        {
            "item": "Wilayah studi",
            "result": STUDY_AREA,
            "interpretation": "Seluruh data penjualan dan UMP diinterpretasikan untuk Provinsi Sulawesi Tenggara.",
        },
        {
            "item": "Model prediktif final",
            "result": selected_predictive_model,
            "interpretation": "Dipilih dari validation WMAPE dengan tie tolerance dan parsimony.",
        },
        {
            "item": "Yearly seasonality model final",
            "result": str(tuned_results[selected_predictive_model].params.get("yearly_fourier_order", 0)),
            "interpretation": "0 berarti nonaktif; nilai di atas 0 adalah Fourier order yang dipilih dari validation set.",
        },
        {
            "item": "WMAPE test model prediktif",
            "result": f"{selected_row['test_WMAPE(%)']:.4f}%",
            "interpretation": "Kesalahan absolut total dibanding total volume aktual.",
        },
        {
            "item": "MAPE test model prediktif",
            "result": f"{selected_row['test_MAPE(%)']:.4f}%",
            "interpretation": "Harus dibaca hati-hati karena aktual sangat kecil pada periode shock.",
        },
        {
            "item": "Target MAPE < 20%",
            "result": "Tercapai" if selected_row["target_mape_below_20_achieved"] else "Belum tercapai",
            "interpretation": "Tidak boleh dinyatakan tercapai jika hasilnya masih di atas 20%.",
        },
        {
            "item": "Model conditional terbaik (deskriptif)",
            "result": best_conditional_model_descriptive,
            "interpretation": "Hanya menunjukkan adaptasi setelah shock teridentifikasi, bukan prediksi shock ex-ante.",
        },
        {
            "item": "UMP Sulawesi Tenggara",
            "result": "2022 Rp2.576.017; 2023 Rp2.758.985; 2024 Rp2.885.964,04; 2025 Rp3.073.551,70",
            "interpretation": "Regressor tahunan dengan empat level pada periode 2022–2025.",
        },
        {
            "item": "Indeks kebijakan CHT",
            "result": "2022 112.00; 2023 123.20; 2024 135.52; 2025 135.52",
            "interpretation": "Indeks kumulatif kebijakan tarif nasional dengan basis 2021 = 100; bukan tarif per batang atau HJE.",
        },
    ]
)

display(executive_summary)

display(
    Markdown(
        f"""
### Kesimpulan komputasional

- Wilayah studi adalah **{STUDY_AREA}**.
- Model prediktif final adalah **`{selected_predictive_model}`**, dipilih hanya dari validation set.
- Yearly seasonality model final menggunakan **fourier_order = {tuned_results[selected_predictive_model].params.get('yearly_fourier_order', 0)}**, dipilih dari validation set.
- Pada test set, model final menghasilkan **WMAPE {selected_row['test_WMAPE(%)']:.4f}%** dan **MAPE {selected_row['test_MAPE(%)']:.4f}%**.
- Model conditional **`{best_conditional_model_descriptive}`** hanya boleh dibahas sebagai model adaptasi setelah shock teridentifikasi.
- UMP menggunakan nilai resmi Sulawesi Tenggara, sedangkan CHT menggunakan proxy perubahan kebijakan nasional karena tarif per batang membutuhkan jenis sigaret, golongan pabrik, dan product mix yang tidak tersedia.
- Koefisien UMP dan proxy CHT harus ditafsirkan terbatas karena hanya ada empat variasi tahunan dan keduanya dapat sangat kollinear.
- Seluruh angka Bab IV, abstrak, dashboard, dan kesimpulan harus diambil dari satu kali **Restart Kernel → Run All** notebook ini.
"""
    )
)
